#Imports

In [1]:
!pip install woodelf_explainer==0.2.14

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 kB 2.2 MB/s eta 0:00:00


In [2]:
import xgboost as xgb
import pandas as pd
import numpy as np
from typing import Union, Dict, Optional, Tuple, Set, List
from math import factorial
import time
from copy import copy
from tqdm import tqdm
from collections import defaultdict
from sklearn.metrics import accuracy_score, f1_score
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
import scipy
import shap

import lightgbm as lgb

# For GPU execution
# import cupy as cp

In [3]:
from woodelf.parse_models import load_decision_tree_ensemble_model
from woodelf.cube_metric import CubeMetric, ShapleyInteractionValues, ShapleyValues

import woodelf


In [4]:
# Useful if you run this on google colab and downloaded the data into your drive.
# If you run the notebook in other environment remove these lines and change the 'pd.read_csv()' function in this notebook to read from
# where you saved you data
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# import woodelfHD from Python file in the drive
!cp /content/drive/MyDrive/ShapResearch/HighDepth/AAAI27/woodelfhd.py /content/

import woodelfhd
from woodelfhd import woodelf_for_high_depth

## Environment Note

Use the CPU enviroment with High-RAM option enabled. It is available for subscribed members for Google Colab (toggle the High-RAM option in Runtime->Change runtime type button).

If you only bought computation units, but not a subscription, you can still run the full notebook using the v2-8-TPU machine. This machine have more then enough RAM and it is pretty cheap. Its CPU is a bit slower though, so you will get a slower runtimes.

Final note: preformance in Colab depends on the machine allocated (the CPU option can allocate several types of machines) and on load balancing inside Colab (when more users use the system running times might be slower). So the running times of our algorithms and the shap python package can very. The difference between our approach and the current state-of-the-art is big enough to be noticed, but the excat running times can be slightly different from the ones stated our paper.

# HIGGs Data Loading and Model training

In [6]:
housing = fetch_california_housing(as_frame=True)

print("california_housing: ")
print(housing.DESCR)

df_housing = housing.frame
df_housing.shape

california_housing: 
.. _california_housing_dataset:

California Housing dataset
--------------------------

**Data Set Characteristics:**

:Number of Instances: 20640

:Number of Attributes: 8 numeric, predictive attributes and the target

:Attribute Information:
    - MedInc        median income in block group
    - HouseAge      median house age in block group
    - AveRooms      average number of rooms per household
    - AveBedrms     average number of bedrooms per household
    - Population    block group population
    - AveOccup      average number of household members
    - Latitude      block group latitude
    - Longitude     block group longitude

:Missing Attribute Values: None

This dataset was obtained from the StatLib repository.
https://www.dcc.fc.up.pt/~ltorgo/Regression/cal_housing.html

The target variable is the median house value for California districts,
expressed in hundreds of thousands of dollars ($100,000).

This dataset was derived from the 1990 U.S. census,

(20640, 9)

In [7]:
housing_train, housing_test, housing_y_train, housing_y_test = train_test_split(
    df_housing[[c for c in df_housing.columns if c != 'MedHouseVal']], df_housing['MedHouseVal'], train_size=0.8, random_state=42
)

housing_train.shape, housing_test.shape

((16512, 8), (4128, 8))

In [8]:
LIGHTGBM_PARAMS = {
    "boosting_type": "gbdt",
    "objective": "regression",
    "metric": "rmse",
    "learning_rate": 0.1,

    # Allow high depth and enough leaves to reach this possible depth
    "num_leaves": 2024,
    "max_depth": 10,
    "min_data_in_leaf": 5,        # Does provide some regulation

    # Sampling (stability + reduces overfit)
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 1,

    # Practical
    "verbosity": -1,
    "seed": 42,
    "force_col_wise": True,            # often faster/safer for wide data
}


def print_model_stat(model, features):
    mobj = load_decision_tree_ensemble_model(model, features)
    depths = {}
    for tree in mobj.trees:
        for leaf, path in tree.get_all_leaves_with_paths():
            depth = len(path)
            if depth not in depths:
                depths[depth] = 0
            depths[depth] += 1

    for depth in sorted(list(depths.keys())):
        print(f"\tPaths of depth {depth}: {depths.get(depth, 0)} paths")

def lightgbm_model(X_train, y_train, params, num_rounds=100):
    train_set = lgb.Dataset(X_train, label=y_train, free_raw_data=False)
    return lgb.train(
        params=params,
        train_set=train_set,
        num_boost_round=num_rounds
    )


def different_depth_lightgbm_models(trainset, y, params, num_rounds, depths):
    models = {}
    for depth in depths:
        new_params = params.copy()
        new_params['max_depth'] = depth
        models[depth] = lightgbm_model(trainset, y, new_params, num_rounds=num_rounds)
        print("\n\n")
        print(f"Trained on depth {depth}")
        print_model_stat(models[depth], list(trainset.columns))
    return models

In [9]:
gbm_100trees = different_depth_lightgbm_models(housing_train, housing_y_train, LIGHTGBM_PARAMS, num_rounds=100, depths=[6,9,12,15,18,21,25,30,40,50,60])




Trained on depth 6
	Paths of depth 2: 5 paths
	Paths of depth 3: 43 paths
	Paths of depth 4: 126 paths
	Paths of depth 5: 372 paths
	Paths of depth 6: 4728 paths



Trained on depth 9
	Paths of depth 2: 4 paths
	Paths of depth 3: 49 paths
	Paths of depth 4: 134 paths
	Paths of depth 5: 377 paths
	Paths of depth 6: 881 paths
	Paths of depth 7: 1919 paths
	Paths of depth 8: 3570 paths
	Paths of depth 9: 15368 paths



Trained on depth 12
	Paths of depth 2: 12 paths
	Paths of depth 3: 47 paths
	Paths of depth 4: 181 paths
	Paths of depth 5: 404 paths
	Paths of depth 6: 902 paths
	Paths of depth 7: 1765 paths
	Paths of depth 8: 3081 paths
	Paths of depth 9: 4951 paths
	Paths of depth 10: 7096 paths
	Paths of depth 11: 9286 paths
	Paths of depth 12: 25132 paths



Trained on depth 15
	Paths of depth 2: 19 paths
	Paths of depth 3: 60 paths
	Paths of depth 4: 217 paths
	Paths of depth 5: 461 paths
	Paths of depth 6: 808 paths
	Paths of depth 7: 1571 paths
	Paths of depth 8: 2814 paths
	Pat

In [10]:
gbm_10trees = different_depth_lightgbm_models(housing_train, housing_y_train, LIGHTGBM_PARAMS, num_rounds=10, depths=[6,9,12,15,18,21])




Trained on depth 6
	Paths of depth 3: 1 paths
	Paths of depth 4: 5 paths
	Paths of depth 5: 13 paths
	Paths of depth 6: 586 paths



Trained on depth 9
	Paths of depth 4: 4 paths
	Paths of depth 5: 17 paths
	Paths of depth 6: 54 paths
	Paths of depth 7: 198 paths
	Paths of depth 8: 416 paths
	Paths of depth 9: 2664 paths



Trained on depth 12
	Paths of depth 4: 3 paths
	Paths of depth 5: 14 paths
	Paths of depth 6: 59 paths
	Paths of depth 7: 191 paths
	Paths of depth 8: 442 paths
	Paths of depth 9: 904 paths
	Paths of depth 10: 1410 paths
	Paths of depth 11: 1967 paths
	Paths of depth 12: 4634 paths



Trained on depth 15
	Paths of depth 4: 3 paths
	Paths of depth 5: 15 paths
	Paths of depth 6: 61 paths
	Paths of depth 7: 186 paths
	Paths of depth 8: 448 paths
	Paths of depth 9: 891 paths
	Paths of depth 10: 1401 paths
	Paths of depth 11: 1921 paths
	Paths of depth 12: 2314 paths
	Paths of depth 13: 2499 paths
	Paths of depth 14: 2497 paths
	Paths of depth 15: 3890 paths



Traine

In [11]:
gbm_1trees = different_depth_lightgbm_models(housing_train, housing_y_train, LIGHTGBM_PARAMS, num_rounds=1, depths=[9, 12,15,18, 21])




Trained on depth 9
	Paths of depth 4: 1 paths
	Paths of depth 6: 4 paths
	Paths of depth 7: 21 paths
	Paths of depth 8: 45 paths
	Paths of depth 9: 274 paths



Trained on depth 12
	Paths of depth 4: 1 paths
	Paths of depth 6: 4 paths
	Paths of depth 7: 21 paths
	Paths of depth 8: 45 paths
	Paths of depth 9: 79 paths
	Paths of depth 10: 152 paths
	Paths of depth 11: 210 paths
	Paths of depth 12: 532 paths



Trained on depth 15
	Paths of depth 4: 1 paths
	Paths of depth 6: 4 paths
	Paths of depth 7: 21 paths
	Paths of depth 8: 45 paths
	Paths of depth 9: 79 paths
	Paths of depth 10: 152 paths
	Paths of depth 11: 210 paths
	Paths of depth 12: 263 paths
	Paths of depth 13: 305 paths
	Paths of depth 14: 285 paths
	Paths of depth 15: 362 paths



Trained on depth 18
	Paths of depth 4: 1 paths
	Paths of depth 6: 4 paths
	Paths of depth 7: 21 paths
	Paths of depth 8: 45 paths
	Paths of depth 9: 79 paths
	Paths of depth 10: 154 paths
	Paths of depth 11: 207 paths
	Paths of depth 12: 265 pa

In [19]:
def high_depth_woodelf_on_models_dict(
        models, consumer_data: pd.DataFrame, background_data: pd.DataFrame, metric: CubeMetric,global_importance: bool = False, GPU=False
    ):
    running_times = {}
    for key, model in models.items():
        start_time = time.time()
        woodelf_for_high_depth(model, consumer_data, background_data, metric=metric, global_importance=global_importance)
        running_time = time.time() - start_time
        running_times[key] = running_time
        print(f"On Depth {key} Took: {running_time}")
    return running_times

In [20]:
def simple_woodelf_on_models_dict(
        models, consumer_data: pd.DataFrame, background_data: pd.DataFrame, metric: CubeMetric, global_importance: bool = False, GPU=False,
    ):
    running_times = {}
    for key, model in models.items():
        start_time = time.time()
        woodelf.simple_woodelf.calculate_background_metric(model, consumer_data, background_data, metric=metric, global_importance=global_importance)
        running_time = time.time() - start_time
        running_times[key] = running_time
        print(f"On Depth {key} Took: {running_time}")
    return running_times

In [21]:
def shap_on_models_dict(models, consumer_data: pd.DataFrame, background_data: pd.DataFrame):
    running_times = {}
    for key, model in models.items():
        start_time = time.time()
        explainer = shap.TreeExplainer(model, background_data, feature_perturbation='interventional')
        simple_shap_values = explainer.shap_values(consumer_data, check_additivity=False)
        running_time = time.time() - start_time
        running_times[key] = running_time
        print(f"On Depth {key} Took: {running_time}")
    return running_times

# Background SHAP

In [15]:
woodelf_running_times = high_depth_woodelf_on_models_dict(
    gbm_100trees,
    consumer_data=housing_test, background_data=housing_train, metric=ShapleyValues(),
    global_importance = False, GPU=False
)
print("Background SHAP: " + " & ".join([str(woodelf_running_times[d]) for d in sorted(list(woodelf_running_times.keys()))]))

Preprocessing the trees and computing SHAP: 100%|██████████| 100/100 [00:01<00:00, 53.03it/s]


M time: 0.0 sec, s time: 0.33 sec (f prepare time: 0.11453771591186523)
On Depth 6 Took: 2.0008363723754883


Preprocessing the trees and computing SHAP: 100%|██████████| 100/100 [00:08<00:00, 11.56it/s]


M time: 0.0 sec, s time: 1.61 sec (f prepare time: 0.5546567440032959)
On Depth 9 Took: 9.293465614318848


Preprocessing the trees and computing SHAP: 100%|██████████| 100/100 [00:20<00:00,  4.78it/s]


...

M time: 0.0 sec, s time: 12.36 sec (f prepare time: 4.232490062713623)
On Depth 25 Took: 70.62711572647095


Preprocessing the trees and computing SHAP: 100%|██████████| 100/100 [01:15<00:00,  1.32it/s]


M time: 0.0 sec, s time: 14.36 sec (f prepare time: 4.904004812240601)
On Depth 30 Took: 82.3209342956543


Preprocessing the trees and computing SHAP:  18%|█▊        | 18/100 [00:16<01:13,  1.12it/s]


TypeError: Cannot cast array data from dtype('uint64') to dtype('int64') according to the rule 'safe'

In [22]:
# TODO delete
woodelf_running_times = high_depth_woodelf_on_models_dict(
    {k: gbm_100trees[k] for k in gbm_100trees if k > 30},
    consumer_data=housing_test, background_data=housing_train, metric=ShapleyValues(),
    global_importance = False, GPU=False
)
print("Background SHAP: " + " & ".join([str(woodelf_running_times[d]) for d in sorted(list(woodelf_running_times.keys()))]))

Preprocessing the trees and computing SHAP: 100%|██████████| 100/100 [01:24<00:00,  1.18it/s]


M time: 0.0 sec, s time: 16.74 sec (f prepare time: 5.711290597915649)
On Depth 40 Took: 92.03525042533875


Preprocessing the trees and computing SHAP: 100%|██████████| 100/100 [01:26<00:00,  1.15it/s]


M time: 0.0 sec, s time: 17.26 sec (f prepare time: 5.8949103355407715)
On Depth 50 Took: 95.4002103805542


Preprocessing the trees and computing SHAP: 100%|██████████| 100/100 [01:26<00:00,  1.15it/s]

M time: 0.0 sec, s time: 17.24 sec (f prepare time: 5.884657621383667)
On Depth 60 Took: 95.26269721984863
Background SHAP: 92.03525042533875 & 95.4002103805542 & 95.26269721984863


In [23]:
woodelf_running_times = simple_woodelf_on_models_dict(
    {6: gbm_100trees[6], 9: gbm_100trees[9], 12: gbm_1trees[12], 15: gbm_1trees[15]}, # Depth 18 crashes due to RAM
    consumer_data=housing_test, background_data=housing_train, metric=ShapleyValues(),
    global_importance = False, GPU=False
)
print("Background SHAP: " + " & ".join([str(woodelf_running_times[d]) for d in sorted(list(woodelf_running_times.keys()))]))

Preprocessing the trees: 100%|██████████| 100/100 [00:01<00:00, 88.86it/s]


cache misses: 237, cache used: 5037, M computation time: 0.33 sec, s computation time: 0.14 sec


Computing the values: 100%|██████████| 100/100 [00:00<00:00, 143.36it/s]


On Depth 6 Took: 1.9375767707824707


Preprocessing the trees: 100%|██████████| 100/100 [14:36<00:00,  8.77s/it]


cache misses: 7466, cache used: 14836, M computation time: 855.92 sec, s computation time: 1.89 sec


Computing the values: 100%|██████████| 100/100 [00:06<00:00, 15.50it/s]


On Depth 9 Took: 883.6507341861725


Preprocessing the trees:  10%|█         | 1/10 [25:41<3:51:17, 1541.94s/it]


KeyboardInterrupt: 

In [24]:
train_sample = housing_train.sample(10, random_state=42)

shap_running_times = shap_on_models_dict(
    {i: gbm_100trees[i] for i in [6,9,12,15,18,21]},
    consumer_data=housing_test, background_data=housing_train,
)
print("Background SHAP: " + " & ".join([str(shap_running_times[d]) for d in sorted(list(shap_running_times.keys()))]))

 95%|=================== | 3931/4128 [00:14<00:00]       

On Depth 6 Took: 13.831302881240845


 98%|===================| 4039/4128 [00:36<00:00]       

On Depth 9 Took: 37.1379599571228


100%|===================| 4115/4128 [01:08<00:00]       

...

On Depth 15 Took: 94.35499811172485


 99%|===================| 4093/4128 [01:55<00:00]       

On Depth 18 Took: 118.43167638778687


 99%|===================| 4101/4128 [02:13<00:00]       

On Depth 21 Took: 137.8219072818756
Background SHAP: 13.831302881240845 & 37.1379599571228 & 68.43456292152405 & 94.35499811172485 & 118.43167638778687 & 137.8219072818756


# Background SHAP IV

In [27]:
woodelf_running_times = high_depth_woodelf_on_models_dict(
    gbm_100trees,
    consumer_data=housing_test, background_data=housing_train, metric=ShapleyInteractionValues(),
    global_importance = False, GPU=False
)
print("Background SHAP IV: " + " & ".join([str(woodelf_running_times[d]) for d in sorted(list(woodelf_running_times.keys()))]))

Preprocessing the trees and computing SHAP: 100%|██████████| 100/100 [00:02<00:00, 47.96it/s]


M time: 0.0 sec, s time: 0.37 sec (f prepare time: 0.11589813232421875)
On Depth 6 Took: 2.201799154281616


Preprocessing the trees and computing SHAP: 100%|██████████| 100/100 [00:10<00:00,  9.82it/s]


M time: 0.01 sec, s time: 1.87 sec (f prepare time: 0.5767545700073242)
On Depth 9 Took: 10.682120561599731


Preprocessing the trees and computing SHAP: 100%|██████████| 100/100 [00:25<00:00,  3.95it/s]


...

M time: 0.01 sec, s time: 19.57 sec (f prepare time: 5.809677600860596)
On Depth 40 Took: 112.38047409057617


Preprocessing the trees and computing SHAP: 100%|██████████| 100/100 [01:48<00:00,  1.08s/it]


M time: 0.01 sec, s time: 20.29 sec (f prepare time: 6.006391525268555)
On Depth 50 Took: 116.47148895263672


Preprocessing the trees and computing SHAP: 100%|██████████| 100/100 [01:48<00:00,  1.08s/it]

M time: 0.01 sec, s time: 20.36 sec (f prepare time: 6.028616428375244)
On Depth 60 Took: 116.93777704238892
Background SHAP IV: 2.201799154281616 & 10.682120561599731 & 26.708813428878784 & 44.65768575668335 & 59.12016844749451 & 70.4153642654419 & 82.32991051673889 & 95.24905180931091 & 112.38047409057617 & 116.47148895263672 & 116.93777704238892


In [28]:
woodelf_running_times = simple_woodelf_on_models_dict(
    {6: gbm_100trees[6], 9: gbm_100trees[9], 12: gbm_1trees[12]}, # depth 15 crashes due to RAM (see the crashes on the fraud data)
    consumer_data=housing_test, background_data=housing_train, metric=ShapleyInteractionValues(),
    global_importance = False, GPU=False
)
print("Background SHAP IV: " + " & ".join([str(woodelf_running_times[d]) for d in sorted(list(woodelf_running_times.keys()))]))

Preprocessing the trees: 100%|██████████| 100/100 [00:01<00:00, 72.14it/s]


cache misses: 237, cache used: 5037, M computation time: 0.46 sec, s computation time: 0.23 sec


Computing the values: 100%|██████████| 100/100 [00:00<00:00, 115.17it/s]


On Depth 6 Took: 2.373610019683838


Preprocessing the trees: 100%|██████████| 100/100 [16:21<00:00,  9.82s/it]


...

On Depth 9 Took: 993.492776632309


Preprocessing the trees: 100%|██████████| 1/1 [18:43<00:00, 1123.65s/it]


cache misses: 558, cache used: 486, M computation time: 1104.35 sec, s computation time: 1.09 sec


Computing the values: 100%|██████████| 1/1 [00:02<00:00,  2.61s/it]

On Depth 12 Took: 1126.6340353488922
Background SHAP IV: 2.373610019683838 & 993.492776632309 & 1126.6340353488922


In [ ]:
# shap Package does not support Background SHAP IV

# Path Dependent SHAP

In [29]:
def pd_high_depth_woodelf_on_models_dict(
        models, consumer_data: pd.DataFrame, metric: CubeMetric, global_importance: bool = False, GPU=False
    ):
    running_times = {}
    for key, model in models.items():
        start_time = time.time()
        woodelf_for_high_depth(model, consumer_data, background_data=None, metric=metric, global_importance=global_importance)
        running_time = time.time() - start_time
        running_times[key] = running_time
        print(f"On Depth {key} Took: {running_time}")
    return running_times

def pd_simple_woodelf_on_models_dict(
        models, consumer_data: pd.DataFrame, metric: CubeMetric, global_importance: bool = False, GPU=False,
    ):
    running_times = {}
    for key, model in models.items():
        start_time = time.time()
        woodelf.simple_woodelf.calculate_path_dependent_metric(model, consumer_data, metric=metric, global_importance=global_importance)
        running_time = time.time() - start_time
        running_times[key] = running_time
        print(f"On Depth {key} Took: {running_time}")
    return running_times

def pd_shap_on_models_dict(models, consumer_data: pd.DataFrame, interaction_values: bool = False):
    running_times = {}
    for key, model in models.items():
        start_time = time.time()
        explainer = shap.TreeExplainer(model, feature_perturbation='tree_path_dependent')
        if interaction_values:
            simple_shap_values = explainer.shap_interaction_values(consumer_data)
        else:
            simple_shap_values = explainer.shap_values(consumer_data, check_additivity=False)
        running_time = time.time() - start_time
        running_times[key] = running_time
        print(f"On Depth {key} Took: {running_time}")
    return running_times

In [30]:
woodelf_running_times = pd_high_depth_woodelf_on_models_dict(
    gbm_100trees,
    consumer_data=housing_test, metric=ShapleyValues(), global_importance = False, GPU=False
)
print("Path-Dependent SHAP: " + " & ".join([str(woodelf_running_times[d]) for d in sorted(list(woodelf_running_times.keys()))]))

Preprocessing the trees and computing SHAP: 100%|██████████| 100/100 [00:01<00:00, 71.56it/s]


M time: 0.0 sec, s time: 0.33 sec (f prepare time: 0.11580610275268555)
On Depth 6 Took: 1.5180015563964844


Preprocessing the trees and computing SHAP: 100%|██████████| 100/100 [00:06<00:00, 14.96it/s]


M time: 0.0 sec, s time: 1.64 sec (f prepare time: 0.5620574951171875)
On Depth 9 Took: 7.226425647735596


Preprocessing the trees and computing SHAP: 100%|██████████| 100/100 [00:16<00:00,  6.02it/s]


...

M time: 0.0 sec, s time: 16.64 sec (f prepare time: 5.772903919219971)
On Depth 40 Took: 73.45054292678833


Preprocessing the trees and computing SHAP: 100%|██████████| 100/100 [01:09<00:00,  1.45it/s]


M time: 0.0 sec, s time: 17.32 sec (f prepare time: 6.013324975967407)
On Depth 50 Took: 77.47286415100098


Preprocessing the trees and computing SHAP: 100%|██████████| 100/100 [01:07<00:00,  1.47it/s]

M time: 0.0 sec, s time: 17.24 sec (f prepare time: 5.985345840454102)
On Depth 60 Took: 76.41584396362305
Path-Dependent SHAP: 1.5180015563964844 & 7.226425647735596 & 18.067182064056396 & 29.20269227027893 & 38.410165309906006 & 46.17642045021057 & 54.07163739204407 & 63.00662159919739 & 73.45054292678833 & 77.47286415100098 & 76.41584396362305


In [31]:
woodelf_running_times = pd_simple_woodelf_on_models_dict(
    {6: gbm_100trees[6], 9: gbm_100trees[9], 12: gbm_1trees[12], 15: gbm_1trees[15]}, # Depth 18 crashes due to RAM (see the crashes on the fraud data)
    consumer_data=housing_test, metric=ShapleyValues(), global_importance = False, GPU=False
)
print("Path-Dependent SHAP: " + " & ".join([str(woodelf_running_times[d]) for d in sorted(list(woodelf_running_times.keys()))]))

Preprocessing the trees: 100%|██████████| 100/100 [00:01<00:00, 53.62it/s]


cache misses: 237, cache used: 5037


Computing the values: 100%|██████████| 100/100 [00:00<00:00, 162.98it/s]


On Depth 6 Took: 2.596339464187622


Preprocessing the trees: 100%|██████████| 100/100 [14:19<00:00,  8.59s/it]


...

On Depth 12 Took: 10679.895780086517


Preprocessing the trees: 100%|██████████| 1/1 [4:15:20<00:00, 15320.42s/it]


cache misses: 1037, cache used: 690


Computing the values: 100%|██████████| 1/1 [00:06<00:00,  6.40s/it]

On Depth 15 Took: 15327.400071382523
Path-Dependent SHAP: 2.596339464187622 & 864.6665213108063 & 10679.895780086517 & 15327.400071382523


In [32]:
shap_running_times = pd_shap_on_models_dict(
    {i: gbm_100trees[i] for i in [6,9,12,15,18,21]}, consumer_data=housing_test,
)
print("Path-Dependent SHAP: " + " & ".join([str(shap_running_times[d]) for d in sorted(list(shap_running_times.keys()))]))

On Depth 6 Took: 0.6915097236633301
On Depth 9 Took: 3.780482053756714
On Depth 12 Took: 9.90656042098999
On Depth 15 Took: 17.396056175231934
On Depth 18 Took: 23.672621250152588
On Depth 21 Took: 28.9932119846344
Path-Dependent SHAP: 0.6915097236633301 & 3.780482053756714 & 9.90656042098999 & 17.396056175231934 & 23.672621250152588 & 28.9932119846344


# Path Dependent SHAP IV

In [33]:
woodelf_running_times = pd_high_depth_woodelf_on_models_dict(
    gbm_100trees,
    consumer_data=housing_test, metric=ShapleyInteractionValues(), global_importance = False, GPU=False
)
print("Path-Dependent SHAP IV: " + " & ".join([str(woodelf_running_times[d]) for d in sorted(list(woodelf_running_times.keys()))]))

Preprocessing the trees and computing SHAP: 100%|██████████| 100/100 [00:01<00:00, 61.14it/s]


M time: 0.0 sec, s time: 0.37 sec (f prepare time: 0.11673569679260254)
On Depth 6 Took: 1.7713167667388916


Preprocessing the trees and computing SHAP: 100%|██████████| 100/100 [00:08<00:00, 12.09it/s]


M time: 0.01 sec, s time: 1.88 sec (f prepare time: 0.5719475746154785)
On Depth 9 Took: 8.847473859786987


Preprocessing the trees and computing SHAP: 100%|██████████| 100/100 [00:21<00:00,  4.67it/s]


...

M time: 0.01 sec, s time: 19.4 sec (f prepare time: 5.8825461864471436)
On Depth 40 Took: 95.0608184337616


Preprocessing the trees and computing SHAP: 100%|██████████| 100/100 [01:28<00:00,  1.13it/s]


M time: 0.01 sec, s time: 20.14 sec (f prepare time: 6.100783348083496)
On Depth 50 Took: 97.8485598564148


Preprocessing the trees and computing SHAP: 100%|██████████| 100/100 [01:31<00:00,  1.10it/s]

M time: 0.01 sec, s time: 20.27 sec (f prepare time: 6.120587587356567)
On Depth 60 Took: 100.27566075325012
Path-Dependent SHAP IV: 1.7713167667388916 & 8.847473859786987 & 22.98124670982361 & 38.34339094161987 & 51.38033938407898 & 62.02243900299072 & 68.57586646080017 & 80.32143783569336 & 95.0608184337616 & 97.8485598564148 & 100.27566075325012


In [34]:
woodelf_running_times = pd_simple_woodelf_on_models_dict(
    {6: gbm_100trees[6], 9: gbm_100trees[9], 12: gbm_10trees[12]}, # depth 15 crashes due to RAM (see the crashes on the fraud data)
    consumer_data=housing_test, metric=ShapleyInteractionValues(), global_importance = False, GPU=False
)
print("Path-Dependent SHAP IV: " + " & ".join([str(woodelf_running_times[d]) for d in sorted(list(woodelf_running_times.keys()))]))

Preprocessing the trees: 100%|██████████| 100/100 [00:00<00:00, 123.11it/s]


cache misses: 237, cache used: 5037


Computing the values: 100%|██████████| 100/100 [00:00<00:00, 130.61it/s]


On Depth 6 Took: 1.6965367794036865


Preprocessing the trees: 100%|██████████| 100/100 [18:55<00:00, 11.35s/it]


...

On Depth 9 Took: 1143.2114748954773


Preprocessing the trees: 100%|██████████| 10/10 [3:25:14<00:00, 1231.47s/it]


cache misses: 5068, cache used: 4556


Computing the values: 100%|██████████| 10/10 [00:13<00:00,  1.36s/it]

On Depth 12 Took: 12329.131623744965
Path-Dependent SHAP IV: 1.6965367794036865 & 1143.2114748954773 & 12329.131623744965


In [35]:
shap_running_times = pd_shap_on_models_dict(
    {6: gbm_100trees[6], 9: gbm_100trees[9], 12: gbm_10trees[12], 15: gbm_1trees[15], 18: gbm_1trees[18], 21: gbm_1trees[21]},
    consumer_data=housing_test, interaction_values=True
)
print("Path-Dependent SHAP IV: " + " & ".join([str(shap_running_times[d]) for d in sorted(list(shap_running_times.keys()))]))

On Depth 6 Took: 22.485955476760864
On Depth 9 Took: 115.09448623657227
On Depth 12 Took: 51.218597412109375
On Depth 15 Took: 9.238362073898315
On Depth 18 Took: 10.916431427001953
On Depth 21 Took: 10.9670991897583
Path-Dependent SHAP IV: 22.485955476760864 & 115.09448623657227 & 51.218597412109375 & 9.238362073898315 & 10.916431427001953 & 10.9670991897583
